In [ ]:
import numpy as np
import matplotlib as mpl
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import os
import pickle
from tqdm.auto import tqdm
from scipy.ndimage import gaussian_filter
import seaborn as sns

from utils_load_data import *

import matplotlib.patches as mpatches
from scipy.stats import wilcoxon
from sklearn.model_selection import KFold
from scipy.stats import sem

from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_predict, cross_val_score
from scipy.stats import pearsonr
from scipy.ndimage import gaussian_filter1d as gaussian_filter
from scipy.stats import gaussian_kde



from sklearn.linear_model import RidgeCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error


## plot config 
mpl.rcdefaults()


save_directory = '' #where the data are saved 


In [ ]:
### PARAMETERS #####

dt = 0.005
t_pre, t_post = 0.2,0.3 #s
dt_trigs = 0 #time in between triggers
time = np.arange(-t_pre, t_post + dt, dt) # time points within psth 
sigma = 1 #smoothing parameter


## 1. Data import

In [ ]:
files_a1 = [ 'NAPOLEON_hs_0','NAPOLEON_hs_1', 'HERCULE_hs_0', 'MMELOIK_hs_1', 'MMELOIK_hs_0', 'FETA_hs_0'] #all A1 files 

n_data_playback = []
f_data_playback = []
neuron_idx_playback = []
for file in files_a1 :
    with open(save_directory + file + f"_playback_{dt}_data_ss", "rb") as fp:
        n_data_s = pickle.load(fp)
        n_data_playback.append(n_data_s[:12])

    with open(save_directory + file + f"_playback_{dt}_feature_ss", "rb") as fp:
        f_data_s = pickle.load(fp)
        f_data_playback.append(f_data_s[:12])

    with open(save_directory+ file + f"_playback_{dt}_neuron_idx_ss", "rb") as fp:
        idx_data_s = pickle.load(fp)
        neuron_idx_playback.append(idx_data_s[:12])

In [ ]:
for n_data_s in n_data_playback :
    n_data_s = smooth_data(n_data_s, sigma = sigma)

n_data_reorganised, f_data_reorganised, neuron_idx_reorganised = re_organise_data(       #to chronologically reorganise the sessions
    n_data_playback, f_data_playback, neuron_idx_playback
) 
n_data_reorganised = [sess / dt for sess in n_data_reorganised] #convert the spk/bin into spk/s

In [ ]:
neuron_idx_reorganised, n_data_reorganised = filter_neurons(files_a1, save_directory, neuron_idx_reorganised, n_data_reorganised) # just to be sure that all analysis have the same units 

# Data format

Shape of n_data_reorganised: n_sessions x units x time_bins. 

because of reorganization, sessions are organized this way: session 0 of animal 1, session 0 of animal 2, session 0 of animal 3, ..., session 11 of animal 1, session 11 of animal 2, session 11 of animal 3. 

the number of units per sessions varies as we have only spike sorted the good channels (selected using the zeta-test).

neuron_idx_reorganised corresponds to the units in n_data_reorganised. This identification is unique and remains the same accross analysis. 

f_data_reorganised contains n_sessions pd dataframe. f_data_reorganised[i] are the features of the session n_data_reorganised[i]

The columns of a dataframe i are the features at each time bin: 
 - Played_frequency: frequency played at this time bin
 - Condition: 0 for tracking, 1 for playback, -1 for warmup or silence
 - Frequency_changes: False if not, True if yes (trigger or not in this time bin)
 - Position: position of the snout
 - Speed_x: speed of the snout (derivative of the position)
 - Mock_frequency: the frequency that should have been played (corresponding to the position) but is not played. In tracking, Mock is equal to 0 (no mock trigger as mock == played). In playback, this column actually carries the mock frequency. 
 - Mock_change: False if not, True if yes (mock trigger or not in this time bin)
 - Block: only useful in the experiments were we have blocks 
 

